<a href="https://colab.research.google.com/github/Jayalekshmi872/ict-aiml-internship/blob/main/multilingual_key_moments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip -q install youtube-transcript-api openpyxl transformers accelerate sentencepiece scikit-learn streamlit
#installs all the Python libraries
#

| Library | Why are we using it in our project? |
|---|---|
| `youtube-transcript-api` | Retrieves available captions or transcripts from YouTube videos. |
| `openpyxl` | Helps Python read and write Excel files (`.xlsx`). |
| `transformers` | Provides pretrained NLP models such as IndicBERT through Hugging Face. |
| `accelerate` | Helps manage and run model training across available hardware, such as GPUs. |
| `sentencepiece` | Provides tokenization support required by some transformer models, including IndicBERT tokenizers. |
| `scikit-learn` | Provides tools for splitting datasets, calculating accuracy, precision, recall, F1-score and other evaluation metrics. |
| `streamlit` | Helps create a web application to display your extracted moments. |

`pipeline.py` is a Python file that contains the functions needed to process YouTube podcast transcripts and identify important conversations.

In [5]:
%%writefile pipeline.py
"""pipeline.py - multilingual key-moment extraction (shared by the notebook and the Streamlit app).

Moments: question (+answer), agreement, disagreement.
Classifier: IndicBERT fine-tuned on (previous sentence, current sentence) pairs.
Fallback when no fine-tuned model is available: multilingual cue-phrase rules.
"""
import re
import numpy as np
import pandas as pd

LABELS = ['neutral', 'question', 'agreement', 'disagreement']

# YouTube caption language codes to try, per language name used in the Excel "language" column
LANG_CODES = {
    'english': ['en', 'en-US', 'en-GB', 'en-IN'], 'hindi': ['hi', 'hi-IN'], 'malayalam': ['ml', 'ml-IN'],
    'tamil': ['ta', 'ta-IN'], 'urdu': ['ur', 'ur-PK'], 'telugu': ['te', 'te-IN'], 'kannada': ['kn', 'kn-IN'],
    'bengali': ['bn', 'bn-BD', 'bn-IN'], 'marathi': ['mr', 'mr-IN'], 'gujarati': ['gu', 'gu-IN'],
    'punjabi': ['pa', 'pa-IN'],
}

CFG = dict(buffer=25, min_len=30, max_len=60, answer_window=20,
           q_thr=0.60, ad_thr=0.70, top_k=8, max_tokens=128)

# '.', '?', '!', Devanagari danda, Urdu full stop, Arabic question mark
END_PUNCT = '.?!\u0964\u06D4\u061F'
STRIP_CHARS = '.,!?;:"\'()[]\u0964\u06D4\u061F\u060C-\u2013\u2014\u2026'


def lang_key(name):
    k = str(name).strip().lower()
    return k if k in LANG_CODES else 'english'


def extract_video_id(url):
    m = re.search(r'(?:youtu\.be/|v=|/embed/|/shorts/)([A-Za-z0-9_-]{11})', str(url))
    return m.group(1) if m else None


# ----------------------------------------------------------------------------------------------
# 1. Transcript (captions only - no video/audio download)
# ----------------------------------------------------------------------------------------------
def fetch_transcript(yt_id, language='english'):
    """Returns (snippets, language_code_used) or (None, None). Works with youtube-transcript-api 0.x and 1.x."""
    from youtube_transcript_api import YouTubeTranscriptApi
    langs = LANG_CODES[lang_key(language)]
    try:
        api = YouTubeTranscriptApi()
        if hasattr(api, 'fetch'):                                   # v1.x
            try:
                fetched = api.fetch(yt_id, languages=langs)
            except Exception:                                       # declared language not found -> any track
                fetched = next(iter(api.list(yt_id))).fetch()
            snips = [{'text': s.text, 'start': s.start, 'duration': s.duration} for s in fetched]
            return snips, getattr(fetched, 'language_code', None)
        return YouTubeTranscriptApi.get_transcript(yt_id, languages=langs), langs[0]   # v0.x
    except Exception as e:
        print(f'  transcript failed for {yt_id}: {type(e).__name__}: {str(e)[:120]}')
        return None, None


# ----------------------------------------------------------------------------------------------
# 2. Caption fragments -> sentences
# ----------------------------------------------------------------------------------------------
def to_words(snips):
    words = []
    for i, s in enumerate(snips):
        txt = re.sub(r'\[[^\]]*\]|\([^)]*\)', ' ', s['text']).replace('\n', ' ')   # [Music], (laughs)
        ws = txt.split()
        if not ws:
            continue
        start, dur = s['start'], s['duration']
        if i + 1 < len(snips):
            dur = min(dur, max(snips[i + 1]['start'] - start, 0.2))
        dur = max(dur, 0.2)
        for k, w in enumerate(ws):
            words.append({'w': w, 't': start + dur * k / len(ws), 'e': start + dur * (k + 1) / len(ws)})
    return words


def build_sentences(snips, max_words=30, pause=1.2):
    words = to_words(snips)
    if not words:
        return [], False
    has_punct = sum(w['w'][-1] in END_PUNCT for w in words) / len(words) > 0.01
    sents, cur, turn = [], [], False

    def flush():
        nonlocal cur, turn
        if cur:
            sents.append({'text': ' '.join(x['w'] for x in cur), 'start': cur[0]['t'],
                          'end': cur[-1]['e'], 'n': len(cur), 'turn': turn})
        cur, turn = [], False

    for w in words:
        tok = w['w']
        if tok.startswith('>>'):                         # speaker-change marker
            flush()
            turn = True
            tok = tok.lstrip('>').strip()
            if not tok:
                continue
            w = {**w, 'w': tok}
        elif cur:
            gap = w['t'] - cur[-1]['e']
            if (cur[-1]['w'][-1] in END_PUNCT or (not has_punct and (gap > pause or len(cur) >= max_words))
                    or len(cur) >= 45):
                flush()
        cur.append(w)
    flush()
    return sents, has_punct


# ----------------------------------------------------------------------------------------------
# 3. Rule-based "weak labels" (used (a) to pre-fill the labelling sheet, (b) as fallback classifier)
#    Matching is token-based (not \b) because \b is unreliable for Indic scripts.
#    >>> These word lists are a starting point - have a native speaker review/extend them. <<<
# ----------------------------------------------------------------------------------------------
EN_WH = r'(what|why|how|when|where|who|whom|whose|which)'
EN_AUX = (r"(do|does|did|is|are|was|were|can|could|would|will|should|have|has|had|am|shall|may|might|"
          r"isn't|aren't|don't|doesn't|didn't|won't)")
EN_Q_START = re.compile(rf'^(so |and |but |okay |ok |well )?({EN_WH}|{EN_AUX})\b', re.I)
EN_AGREE = re.compile('|'.join([
    r"i agree", r"i totally agree", r"couldn'?t agree more", r"exactly", r"absolutely", r"you'?re (absolutely )?right",
    r"you are right", r"that'?s (so )?(true|right|correct)", r"definitely", r"for sure", r"well said",
    r"fair (point|enough)", r"i think so too", r"same (here|for me)", r"100 ?%", r"makes sense", r"^yes\b",
    r"^yeah\b", r"^true\b", r"i feel the same"]), re.I)
EN_DISAGREE = re.compile('|'.join([
    r"i disagree", r"i don'?t agree", r"i don'?t think so", r"not really", r"that'?s not (true|right|correct|it)",
    r"i wouldn'?t say", r"i'?m not sure about that", r"on the contrary", r"i beg to differ", r"not necessarily",
    r"i don'?t buy", r"that'?s wrong", r"^no\b", r"^nope\b", r"^but\b", r"^actually\b", r"^however\b",
    r"hold on", r"i('d| would) push back", r"i don'?t think"]), re.I)

LEX = {
    'hindi': {
        'agree': ['बिल्कुल', 'बिलकुल', 'सही कहा', 'ठीक कहा', 'मैं सहमत हूँ', 'मैं सहमत हूं', 'सही है', 'सच है',
                  'हाँ', 'हां', 'जी हाँ', 'जी हां', 'बिल्कुल सही'],
        'disagree': ['सहमत नहीं', 'असहमत', 'नहीं', 'ऐसा नहीं है', 'ऐसा नहीं', 'मुझे नहीं लगता', 'लेकिन', 'पर',
                     'गलत', 'ग़लत'],
        'question': ['क्या', 'कैसे', 'क्यों', 'कब', 'कहाँ', 'कहां', 'कौन', 'कितना', 'कितने', 'किस', 'किसने']},
    'urdu': {
        'agree': ['بالکل', 'جی ہاں', 'جی', 'ٹھیک ہے', 'درست', 'میں متفق ہوں', 'آپ صحیح کہہ رہے ہیں', 'صحیح کہا'],
        'disagree': ['متفق نہیں', 'اختلاف', 'نہیں', 'ایسا نہیں ہے', 'ایسا نہیں', 'مجھے نہیں لگتا', 'لیکن', 'غلط'],
        'question': ['کیا', 'کیسے', 'کیوں', 'کب', 'کہاں', 'کون', 'کتنا', 'کتنے', 'کس']},
    'tamil': {
        'agree': ['ஆமாம்', 'ஆமா', 'கண்டிப்பா', 'கண்டிப்பாக', 'நிச்சயமா', 'நிச்சயமாக', 'உண்மைதான்', 'சரிதான்',
                  'சரியா சொன்னீங்க', 'நான் ஒத்துக்கிறேன்'],
        'disagree': ['இல்லை', 'இல்ல', 'ஆனா', 'ஆனால்', 'தப்பு', 'தவறு', 'நான் ஒத்துக்கல', 'எனக்கு அப்படி தோணல'],
        'question': ['என்ன', 'எப்படி', 'ஏன்', 'எப்போ', 'எப்போது', 'எங்கே', 'எங்க', 'யார்', 'எவ்வளவு']},
    'malayalam': {
        'agree': ['അതെ', 'ശരിയാണ്', 'ശരിയാ', 'സത്യമാണ്', 'തീർച്ചയായും', 'ഞാൻ യോജിക്കുന്നു'],
        'disagree': ['അല്ല', 'ഇല്ല', 'പക്ഷേ', 'പക്ഷെ', 'തെറ്റാണ്', 'ഞാൻ യോജിക്കുന്നില്ല', 'എനിക്ക് തോന്നുന്നില്ല'],
        'question': ['എന്ത്', 'എന്താണ്', 'എങ്ങനെ', 'എന്തുകൊണ്ട്', 'എപ്പോൾ', 'എവിടെ', 'ആര്', 'എത്ര', 'ആണോ']},
}


def _clean_tokens(text, n=None):
    toks = [t.strip(STRIP_CHARS) for t in text.split()]
    toks = [t for t in toks if t]
    return toks[:n] if n else toks


def _first_hit(text, phrases, head=14):
    """Earliest position (token index) where any phrase appears inside the first `head` tokens, else None."""
    hay_t = _clean_tokens(text, head)
    hay = ' ' + ' '.join(hay_t) + ' '
    best = None
    for p in phrases:
        idx = hay.find(' ' + p + ' ')
        if idx >= 0:
            pos = hay[:idx + 1].count(' ')
            best = pos if best is None else min(best, pos)
    return best


def weak_label(cur, lang, has_punct, n_words=None):
    """Rule-based label for a sentence: neutral / question / agreement / disagreement."""
    key = lang_key(lang)
    n = n_words if n_words is not None else len(cur.split())
    if '?' in cur or '\u061F' in cur:
        return 'question'
    if key == 'english':
        if not has_punct and EN_Q_START.match(cur.strip()) and 3 <= n <= 28:
            return 'question'
        head = ' '.join(cur.split()[:14])
        a, d = EN_AGREE.search(head), EN_DISAGREE.search(head)
        if a and (not d or a.start() < d.start()):
            return 'agreement'
        if d:
            return 'disagreement'
        return 'neutral'
    lex = LEX.get(key)
    if lex is None:
        return 'neutral'
    toks = set(_clean_tokens(cur))
    if not has_punct and 3 <= n <= 25 and any(q in toks for q in lex['question']):
        return 'question'
    a, d = _first_hit(cur, lex['agree']), _first_hit(cur, lex['disagree'])
    if d is not None and (a is None or d <= a):        # tie -> disagreement ("सहमत नहीं" contains the agree stem)
        return 'disagreement'
    if a is not None:
        return 'agreement'
    return 'neutral'


# ----------------------------------------------------------------------------------------------
# 4. Fine-tuned IndicBERT classifier
# ----------------------------------------------------------------------------------------------
class MomentClassifier:
    """Loads a fine-tuned sequence-pair classifier (local folder or Hugging Face repo id)."""

    def __init__(self, path):
        import torch
        from transformers import AutoTokenizer, AutoModelForSequenceClassification
        self.torch = torch
        self.tok = AutoTokenizer.from_pretrained(path)
        self.model = AutoModelForSequenceClassification.from_pretrained(path).eval()
        self.dev = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.model.to(self.dev)
        id2 = self.model.config.id2label
        self.labels = [id2[i] for i in range(len(id2))]

    def predict(self, pairs, bs=32):
        out = []
        with self.torch.no_grad():
            for i in range(0, len(pairs), bs):
                chunk = pairs[i:i + bs]
                enc = self.tok([p for p, _ in chunk], [c for _, c in chunk], truncation=True,
                               max_length=CFG['max_tokens'], padding=True, return_tensors='pt').to(self.dev)
                out.append(self.torch.softmax(self.model(**enc).logits, -1).cpu().numpy())
        return np.vstack(out) if out else np.zeros((0, len(self.labels)))


def classify_sentences(sents, has_punct, lang, clf=None):
    """Returns (labels, confidences) per sentence. Uses the fine-tuned model if given, else the rules."""
    if clf is not None:
        pairs = [(sents[i - 1]['text'] if i else '', s['text']) for i, s in enumerate(sents)]
        P = clf.predict(pairs)
        idx = P.argmax(1)
        return [clf.labels[k] for k in idx], [float(P[i, k]) for i, k in enumerate(idx)]
    labs = [weak_label(s['text'], lang, has_punct, s['n']) for s in sents]
    confs = [1.0 if l == 'neutral' else 0.8 if l == 'question' else 0.75 for l in labs]
    return labs, confs


# ----------------------------------------------------------------------------------------------
# 5. Moments -> segments [x - y, x + y]
# ----------------------------------------------------------------------------------------------
def find_moments(sents, labs, confs, cfg=CFG):
    out = []
    for i, s in enumerate(sents):
        if labs[i] == 'question' and confs[i] >= cfg['q_thr']:
            ans, j, words = [], i + 1, 0
            while j < len(sents) and len(ans) < 3:
                nx = sents[j]
                if nx['start'] - s['end'] > (cfg['answer_window'] if not ans else 4):
                    break
                if labs[j] == 'question':
                    break
                ans.append(nx)
                words += nx['n']
                j += 1
            if words >= 4:
                conf = min(1.0, confs[i] + 0.1 * ans[0]['turn'])
                out.append({'type': 'Question-Answer', 'm_start': s['start'], 'm_end': ans[-1]['end'],
                            'conf': round(conf, 2),
                            'text': s['text'] + '  ->  ' + ' '.join(a['text'] for a in ans)})
        elif (labs[i] in ('agreement', 'disagreement') and confs[i] >= cfg['ad_thr'] and i > 0
              and s['n'] >= 2 and s['start'] - sents[i - 1]['end'] < 8):
            out.append({'type': labs[i].capitalize(), 'm_start': sents[i - 1]['start'], 'm_end': s['end'],
                        'conf': round(confs[i], 2), 'text': sents[i - 1]['text'] + '  ->  ' + s['text']})
    return out


def make_segment(m_start, m_end, y, video_len, cfg=CFG):
    x = (m_start + m_end) / 2
    half = max(y, (m_end - m_start) / 2 + 2)
    half = min(max(half, cfg['min_len'] / 2), cfg['max_len'] / 2)
    s, e = x - half, x + half
    if s < 0:
        e, s = e - s, 0.0
    if video_len and e > video_len:
        s, e = max(0.0, s - (e - video_len)), video_len
    return s, e, x


def _overlap(a, b):
    inter = max(0, min(a[1], b[1]) - max(a[0], b[0]))
    return inter / max(1e-9, min(a[1] - a[0], b[1] - b[0]))


def select_segments(moments, y, video_len, cfg=CFG):
    segs = []
    for m in moments:
        s, e, x = make_segment(m['m_start'], m['m_end'], y, video_len, cfg)
        segs.append({**m, 'x': x, 'seg_start': s, 'seg_end': e})
    segs.sort(key=lambda r: -r['conf'])
    kept = []
    for r in segs:
        if any(k['type'] == r['type'] and _overlap((k['seg_start'], k['seg_end']), (r['seg_start'], r['seg_end'])) > 0.5
               for k in kept):
            continue
        if sum(k['type'] == r['type'] for k in kept) >= cfg['top_k']:
            continue
        kept.append(r)
    return sorted(kept, key=lambda r: r['seg_start'])


def fmt(t):
    t = int(t)
    return f'{t // 3600:02d}:{(t % 3600) // 60:02d}:{t % 60:02d}'


def embed_url(yt_id, s, e):
    return f'https://www.youtube.com/embed/{yt_id}?start={int(s)}&end={int(e)}&rel=0'


def yt_embed_html(yt_id, s, e, w=560, h=315, autoplay=0):
    """Iframe with the attributes YouTube's own embed code uses (referrerpolicy avoids 'Error 153' in sandboxes)."""
    return (f'<iframe width="{w}" height="{h}" src="{embed_url(yt_id, s, e)}&autoplay={autoplay}" '
            'title="YouTube video player" frameborder="0" '
            'allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture; web-share" '
            'referrerpolicy="strict-origin-when-cross-origin" allowfullscreen></iframe>')


def analyse(yt_id, language='english', title='', video_id='', clf=None, cfg=CFG, snips=None):
    """Full pipeline for one video. Returns (DataFrame of segments, info dict)."""
    lang_used = None
    if snips is None:
        snips, lang_used = fetch_transcript(yt_id, language)
    if not snips:
        return pd.DataFrame(), {'error': 'no transcript'}
    sents, has_punct = build_sentences(snips)
    video_len = snips[-1]['start'] + snips[-1]['duration']
    labs, confs = classify_sentences(sents, has_punct, language, clf)
    segs = select_segments(find_moments(sents, labs, confs, cfg), cfg['buffer'], video_len, cfg)
    rows = []
    for r in segs:
        in_seg = ' '.join(s['text'] for s in sents if s['start'] >= r['seg_start'] and s['end'] <= r['seg_end'])
        rows.append({'video_id': video_id, 'title': title, 'yt_id': yt_id, 'language': language, 'type': r['type'],
                     'moment_time': fmt(r['x']), 'seg_start': round(r['seg_start'], 1),
                     'seg_end': round(r['seg_end'], 1), 'duration_s': round(r['seg_end'] - r['seg_start'], 1),
                     'confidence': r['conf'], 'key_exchange': r['text'][:400], 'segment_transcript': in_seg,
                     'watch_url': f"https://www.youtube.com/watch?v={yt_id}&t={int(r['seg_start'])}s",
                     'embed_url': embed_url(yt_id, r['seg_start'], r['seg_end'])})
    info = {'fragments': len(snips), 'sentences': len(sents), 'punctuated': has_punct, 'lang_used': lang_used,
            'video_len': video_len, 'label_counts': pd.Series(labs).value_counts().to_dict()}
    return pd.DataFrame(rows), info


# ----------------------------------------------------------------------------------------------
# 6. Training-data helper (weak labels -> sheet for human review)
# ----------------------------------------------------------------------------------------------
def build_training_candidates(stores, neutral_ratio=1.5, seed=0):
    """stores: list of dicts {video_id, language, sents, has_punct}. Returns a DataFrame to review."""
    rows = []
    for st in stores:
        S = st['sents']
        for i, s in enumerate(S):
            rows.append({'video_id': st['video_id'], 'language': st['language'], 'start': round(s['start'], 1),
                         'prev': S[i - 1]['text'] if i else '', 'cur': s['text'],
                         'weak_label': weak_label(s['text'], st['language'], st['has_punct'], s['n'])})
    df = pd.DataFrame(rows)
    pos = df[df.weak_label != 'neutral']
    neu = df[df.weak_label == 'neutral']
    neu = neu.sample(min(len(neu), int(len(pos) * neutral_ratio) + 50), random_state=seed)
    out = pd.concat([pos, neu]).sort_values(['video_id', 'start']).reset_index(drop=True)
    out['label'] = ''          # <- fill / correct this column by hand (leave blank to accept weak_label)
    return out

Overwriting pipeline.py


It handles:
1. Retrieving YouTube captions.
2. Converting captions into sentences.
3. Generating preliminary labels.
4. Using the fine-tuned IndicBERT model for classification.
5. Identifying Question–Answer, Agreement and Disagreement moments.
6. Generating video clip timestamps and links.
7. Preparing training data for human review.

## 2. Configuration and Excel input
Upload `video_metadata.xlsx` (columns: `video_id, video_url, video_title, language, ...`). Rows with an empty `video_url` are skipped.

In [6]:
import pandas as pd, json, os, time, numpy as np
from IPython.display import HTML, display
import pipeline as pl

EXCEL_PATH = 'video_metadata.xlsx'
pl.CFG.update(buffer=25,          # y: segment = [x - y, x + y]
              min_len=30, max_len=60,   # (fixed: the earlier file had 'MIN_LEN 30' without '=')
              answer_window=20,   # answer must start within 20 s of the question
              q_thr=0.60, ad_thr=0.70, top_k=8)

videos = pd.read_excel(EXCEL_PATH)
videos['yt_id'] = videos['video_url'].apply(pl.extract_video_id)
skipped = videos[videos.yt_id.isna()]
if len(skipped): print('Skipping rows without a valid link:', list(skipped.video_id))
videos = videos[videos.yt_id.notna()].reset_index(drop=True)
print(videos.groupby('language').size())
videos[['video_id','video_title','language','yt_id']]

language
English      5
Hindi        2
Malayalam    2
Tamil        2
Urdu         1
dtype: int64


,video_id,video_title,language,yt_id
0,V001,confidence isn't loud,English,rKpltaOMFdc
1,V002,Does money buy happiness,English,bdpyQm5l78o
2,V003,You cannot always be positive,English,-pYOlucclio
3,V004,We the women,English,63mazZL-Lh4
4,V005,The future mark zuckerberg is trying to build,English,oX7OduG1YmI
5,V006,Bollywood darkside nobody talk about,Hindi,EJXPeOKPG3Y
6,V007,Brain Rot is not Real,Hindi,u25wf-qmF0E
7,V008,Your mind is Easier to Fool than you think,Malayalam,cb_6ExZNwFc
8,V009,"Time travel,Multiverse & Interstellar",Malayalam,ViwK-gtSLgk
9,V010,Before you give up watch this!,Tamil,zEEP20TG43w


## 3. Fetch captions (no video download) and build sentences
Captions are requested in the language written in the Excel `language` column. Results are cached in `transcripts.json` so re-runs are quick.

In [7]:
CACHE = 'transcripts.json'
cache = json.load(open(CACHE, encoding='utf-8')) if os.path.exists(CACHE) else {}
stores = []
for r in videos.itertuples():
    if r.yt_id not in cache:
        snips, used = pl.fetch_transcript(r.yt_id, r.language)
        cache[r.yt_id] = {'snips': snips, 'lang_used': used}
        json.dump(cache, open(CACHE, 'w', encoding='utf-8'), ensure_ascii=False)
    snips, used = cache[r.yt_id]['snips'], cache[r.yt_id]['lang_used']
    if not snips: print(f'{r.video_id} ({r.language}): NO CAPTIONS - see Whisper note at the end'); continue
    sents, hp = pl.build_sentences(snips)
    stores.append(dict(video_id=r.video_id, language=r.language, yt_id=r.yt_id, title=r.video_title, snips=snips,
                       sents=sents, has_punct=hp, lang_used=used))
    flag = '' if (used is None or used.lower().startswith(pl.LANG_CODES[pl.lang_key(r.language)][0])) else f'  <-- WARNING: captions are "{used}"'
    print(f'{r.video_id} {r.language:10s} | {len(snips):5d} fragments -> {len(sents):4d} sentences | punctuated={hp} | caption lang={used}{flag}')
# sanity check that Indic text came through correctly
for s in stores:
    print(s['language'], '|', s['sents'][len(s['sents'])//2]['text'][:90])

V001 English    |  1412 fragments ->  361 sentences | punctuated=False | caption lang=en
V002 English    |  1104 fragments ->  287 sentences | punctuated=False | caption lang=en
V003 English    |  1136 fragments ->  276 sentences | punctuated=False | caption lang=en
V004 English    |   697 fragments ->  387 sentences | punctuated=True | caption lang=en
V005 English    |   477 fragments ->  326 sentences | punctuated=True | caption lang=en
V006 Hindi      |  1344 fragments ->  649 sentences | punctuated=True | caption lang=hi
V007 Hindi      |  1959 fragments -> 1104 sentences | punctuated=True | caption lang=hi
V008 Malayalam  |  3984 fragments ->  658 sentences | punctuated=True | caption lang=ml
V009 Malayalam  |  2550 fragments ->  359 sentences | punctuated=True | caption lang=ml
V010 Tamil      |  2892 fragments -> 1218 sentences | punctuated=True | caption lang=ta
V011 Tamil      |  2047 fragments ->  734 sentences | punctuated=True | caption lang=ta
V012 Urdu       |  1164 fragm

## 4. Build the labelled dataset for fine-tuning
Fine-tuning needs labelled examples. The cell below creates `labels_to_review.csv` with a **rule-based `weak_label`** for each (previous sentence, current sentence) pair, balanced so that *neutral* does not dominate.

**You must review it** (Excel / Google Sheets): put the correct class in the `label` column - `neutral`, `question`, `agreement` or `disagreement` (leave blank to accept `weak_label`). Aim to check at least a few hundred rows, covering every language. If you accept all weak labels, the fine-tuned model just learns the cue-word rules and will not beat them - the human correction is what makes fine-tuning worthwhile. Ask a native speaker to check Hindi/Malayalam/Tamil/Urdu rows if you can.

Then upload the corrected file as `labels_reviewed.csv`.

In [8]:
cand = pl.build_training_candidates(stores, neutral_ratio=1.5)
cand.to_csv('labels_to_review.csv', index=False, encoding='utf-8-sig')
print(cand.groupby(['language','weak_label']).size().unstack(fill_value=0))
try:
    from google.colab import files; files.download('labels_to_review.csv')
except Exception: pass
cand.head(10)

weak_label  agreement  disagreement  neutral  question
language                                              
English            75            24      398        52
Hindi              64           204      365       162
Malayalam          27           107      231        52
Tamil              45           138      475        49
Urdu                0             0       79         0


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

,video_id,language,start,prev,cur,weak_label,label
0,V001,English,37.4,is the CEO of speaker by nimi meta she's a TV ...,uh the the beautiful nimi meta on the podcast ...,neutral,
1,V001,English,82.5,that's important context to understand kind of...,for people did journalism at University came s...,neutral,
2,V001,English,106.0,wow it's really going to be doggy dog and then...,online competition um you had to submit a vide...,neutral,
3,V001,English,123.2,I need you to be extras and we were in the dre...,her hair done by her friend and you know peopl...,neutral,
4,V001,English,131.6,her hair done by her friend and you know peopl...,I'm like I really produced directed and presen...,neutral,
5,V001,English,251.2,marathon right it's not the Finish Line it's j...,is because it helps you enjoy the process whic...,neutral,
6,V001,English,259.7,is because it helps you enjoy the process whic...,in a different way to what you think you actua...,neutral,
7,V001,English,268.3,in a different way to what you think you actua...,out all the way you realize that's actually wh...,neutral,
8,V001,English,283.8,job or you know reaching this point was going ...,now what's the next part of life so I think it...,neutral,
9,V001,English,291.7,now what's the next part of life so I think it...,just markers of what's next yeah and and the t...,neutral,


In [9]:
# --- after reviewing: upload labels_reviewed.csv (Colab: left sidebar > Files > upload) ---
if os.path.exists('labels_reviewed.csv'):
    lab = pd.read_csv('labels_reviewed.csv', encoding='utf-8-sig').fillna({'prev': '', 'label': ''})
    lab['label'] = lab['label'].astype(str).str.strip().str.lower()
    lab.loc[lab.label.isin(['', 'nan']), 'label'] = lab['weak_label']
    print('Using human-reviewed labels')
else:
    lab = cand.copy(); lab['label'] = lab['weak_label']
    print('WARNING: labels_reviewed.csv not found -> training on WEAK labels only (demo run).')
assert set(lab.label) <= set(pl.LABELS), set(lab.label) - set(pl.LABELS)
lab['cur'] = lab['cur'].astype(str); lab['prev'] = lab['prev'].astype(str)
print(lab.groupby(['language', 'label']).size().unstack(fill_value=0))

label      agreement  disagreement  neutral  question
language                                             
English           75            24      398        52
Hindi             64           204      365       162
Malayalam         27           107      231        52
Tamil             45           138      475        49
Urdu               0             0       79         0


## 5. Time-block split (no leakage)
Neighbouring sentences are near-duplicates in context, so a random split would leak. Within each video, the **first 70 %** of the timeline is train, the next **15 %** validation, the last **15 %** test.

In [10]:
lab['pos'] = lab.groupby('video_id')['start'].rank(pct=True)
train_df = lab[lab.pos <= 0.70]; val_df = lab[(lab.pos > 0.70) & (lab.pos <= 0.85)]; test_df = lab[lab.pos > 0.85]
for n, d in [('train', train_df), ('val', val_df), ('test', test_df)]:
    print(n, len(d), d.label.value_counts().to_dict())

train 1778 {'neutral': 1077, 'disagreement': 326, 'question': 215, 'agreement': 160}
val 381 {'neutral': 235, 'disagreement': 69, 'question': 52, 'agreement': 25}
test 388 {'neutral': 236, 'disagreement': 78, 'question': 48, 'agreement': 26}


## 6. Fine-tune IndicBERT v2 (`ai4bharat/IndicBERTv2-MLM-only`)
Input = *(previous sentence, current sentence)*, output = 4 classes. Use a **GPU runtime** in Colab (Runtime > Change runtime type > T4).

In [11]:
import torch
import time
from torch.utils.data import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer,
                          DataCollatorWithPadding)
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

BASE = 'ai4bharat/IndicBERTv2-MLM-only'      # covers English + 22 Indian languages incl. Hindi, Tamil, Malayalam, Urdu
OUT  = 'models/indicbert_moments'
lab2id = {l: i for i, l in enumerate(pl.LABELS)}; id2lab = {i: l for l, i in lab2id.items()}
tok = AutoTokenizer.from_pretrained(BASE)

class PairDS(Dataset):
    def __init__(self, d):
        self.enc = tok(list(d.prev), list(d.cur), truncation=True, max_length=pl.CFG['max_tokens'])
        self.y = [lab2id[l] for l in d.label]
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}; item['labels'] = self.y[i]; return item

def compute_metrics(p):
    pred = p.predictions.argmax(-1)
    return {'macro_f1': f1_score(p.label_ids, pred, average='macro'), 'accuracy': accuracy_score(p.label_ids, pred)}

tr_ds, va_ds, te_ds = PairDS(train_df), PairDS(val_df), PairDS(test_df)
model = AutoModelForSequenceClassification.from_pretrained(BASE, num_labels=len(pl.LABELS), id2label=id2lab, label2id=lab2id)

# Calculate total steps for 4 epochs: (len(tr_ds) // batch_size) * epochs
batch_size = 16
total_steps = (len(tr_ds) // batch_size) * 4
warmup_steps = int(0.1 * total_steps)

args = TrainingArguments(output_dir='ckpt', num_train_epochs=4, per_device_train_batch_size=batch_size,
                         per_device_eval_batch_size=32, learning_rate=3e-5, weight_decay=0.01, warmup_steps=warmup_steps,
                         eval_strategy='epoch', save_strategy='epoch', load_best_model_at_end=True,
                         metric_for_best_model='macro_f1', save_total_limit=1, fp16=torch.cuda.is_available(),
                         report_to='none', seed=42)
trainer = Trainer(model=model, args=args, train_dataset=tr_ds, eval_dataset=va_ds,
                  data_collator=DataCollatorWithPadding(tok), compute_metrics=compute_metrics)
t0 = time.time(); trainer.train(); print(f'training time: {time.time()-t0:.0f}s')
trainer.save_model(OUT); tok.save_pretrained(OUT)
print('saved to', OUT)

config.json:   0%|          | 0.00/639 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/51.0 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 7.75MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.12GB            

... (9 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

training time: 221s


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved to models/indicbert_moments


## 7. Evaluate on the held-out test block (per class and per language)

In [12]:
pred = trainer.predict(te_ds); yp = pred.predictions.argmax(-1); yt = pred.label_ids
print(classification_report(yt, yp, labels=list(range(4)), target_names=pl.LABELS, digits=3, zero_division=0))
print(pd.DataFrame(confusion_matrix(yt, yp, labels=list(range(4))), index=pl.LABELS, columns=pl.LABELS))
t = test_df.assign(ok=(yt == yp))
print(t.groupby('language').apply(lambda g: pd.Series({'n': len(g), 'accuracy': g.ok.mean()})))
# Baseline = the rule-based weak label itself. Only meaningful if labels were human-reviewed.
print('rule baseline macro-F1 :', round(f1_score(test_df.label, test_df.weak_label, average='macro', labels=pl.LABELS, zero_division=0), 3))
print('fine-tuned macro-F1    :', round(f1_score(yt, yp, average='macro', labels=list(range(4)), zero_division=0), 3))

              precision    recall  f1-score   support

     neutral      0.929     0.890     0.909       236
    question      0.980     1.000     0.990        48
   agreement      0.818     0.692     0.750        26
disagreement      0.747     0.872     0.805        78

    accuracy                          0.887       388
   macro avg      0.869     0.863     0.863       388
weighted avg      0.891     0.887     0.887       388

              neutral  question  agreement  disagreement
neutral           210         1          3            22
question            0        48          0             0
agreement           7         0         18             1
disagreement        9         0          1            68
               n  accuracy
language                  
English     85.0  0.882353
Hindi      120.0  0.891667
Malayalam   64.0  0.828125
Tamil      107.0  0.925234
Urdu        12.0  0.833333
rule baseline macro-F1 : 1.0
fine-tuned macro-F1    : 0.863


/tmp/ipykernel_376/1587238288.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  print(t.groupby('language').apply(lambda g: pd.Series({'n': len(g), 'accuracy': g.ok.mean()})))


## 8. Extract moments from every video with the fine-tuned model

In [13]:
clf = pl.MomentClassifier(OUT)
frames = []
for r in videos.itertuples():
    snips = cache[r.yt_id]['snips']
    if not snips: continue
    df, info = pl.analyse(r.yt_id, r.language, r.video_title, r.video_id, clf=clf, snips=snips)
    print(r.video_id, r.language, info['label_counts'])
    frames.append(df)
results = pd.concat([f for f in frames if len(f)], ignore_index=True)
results.to_csv('extracted_moments.csv', index=False, encoding='utf-8-sig')
print(results.groupby(['language', 'type']).size().unstack(fill_value=0))
results[['video_id','language','type','moment_time','seg_start','seg_end','confidence','key_exchange']]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

V001 English {'neutral': 351, 'agreement': 8, 'disagreement': 2}
V002 English {'neutral': 264, 'agreement': 21, 'disagreement': 2}
V003 English {'neutral': 249, 'agreement': 26, 'disagreement': 1}
V004 English {'neutral': 306, 'agreement': 30, 'question': 29, 'disagreement': 22}
V005 English {'neutral': 277, 'question': 24, 'agreement': 22, 'disagreement': 3}
V006 Hindi {'neutral': 465, 'disagreement': 113, 'agreement': 39, 'question': 32}
V007 Hindi {'neutral': 747, 'disagreement': 175, 'question': 138, 'agreement': 44}
V008 Malayalam {'neutral': 467, 'disagreement': 117, 'question': 44, 'agreement': 30}
V009 Malayalam {'neutral': 253, 'disagreement': 79, 'agreement': 20, 'question': 7}
V010 Tamil {'neutral': 972, 'disagreement': 175, 'question': 38, 'agreement': 33}
V011 Tamil {'neutral': 651, 'disagreement': 55, 'agreement': 16, 'question': 12}
V012 Urdu {'neutral': 314, 'disagreement': 9, 'agreement': 2}
type       Agreement  Disagreement  Question-Answer
language                  

,video_id,language,type,moment_time,seg_start,seg_end,confidence,key_exchange
0,V001,English,Agreement,00:04:36,251.0,301.0,0.76,out all the way you realize that's actually wh...
1,V001,English,Agreement,00:10:47,622.3,672.3,0.83,really great reframe for me to just keep going...
2,V001,English,Agreement,00:11:49,684.6,734.6,0.77,and we consume so much stuff that just sounds ...
3,V001,English,Agreement,00:25:10,1485.8,1535.8,0.79,the groundwork in identifying what that voice ...
4,V001,English,Agreement,00:35:03,2078.7,2128.7,0.80,where I should have probably stood up for myse...
...,...,...,...,...,...,...,...,...
204,V012,Urdu,Disagreement,00:27:53,1648.9,1698.9,0.75,हैं ना कि अंग्रेजी में एक्सपोजर उर्दू का वो एक...
205,V012,Urdu,Disagreement,00:32:07,1902.6,1952.6,0.85,जिसमें उनके ख्यालात पैदा होते हैं उस जुबान को ...
206,V012,Urdu,Disagreement,00:44:59,2674.2,2724.2,0.75,पूरा एक लुत्फ है उस वक्त हमें फारसी कलाम जाहिर...
207,V012,Urdu,Agreement,00:49:41,2956.5,3006.5,0.94,आम जिंदगी की जो हमारी एक मादी जरूरियत है उससे ...


## 9. Playing the segment in Colab
**Why it sometimes opens/plays on YouTube instead of inside Colab**
- A plain `watch?v=...&t=120s` link starts at 120 s but **never stops** - a watch link cannot carry an end time. Only an *embed* (`start=` and `end=`) or a cut clip stops at the segment end.
- Colab *can* show YouTube embeds, but YouTube refuses to play an embed when (a) the uploader disabled embedding, or (b) the page sends no referrer (shows "Video unavailable" / *Error 153*). The helper below uses YouTube's own embed attributes (including `referrerpolicy`), which fixes (b) in most cases. If a video still shows "Watch on YouTube", it is case (a) - nothing in Colab can change that.
- Most reliable alternatives: the Streamlit app (section 11) or downloading only the 30-60 s clip (below; YouTube sometimes blocks downloads from Colab's IPs).

In [14]:
def play(i, autoplay=0):
    r = results.iloc[i]
    print(f"{r.type} | {r.title} | {r.language} | {r.seg_start}s -> {r.seg_end}s | conf {r.confidence}")
    print(r.key_exchange)
    display(HTML(pl.yt_embed_html(r.yt_id, r.seg_start, r.seg_end, autoplay=autoplay)))
    print('Open on YouTube (starts here, does not stop):', r.watch_url)
play(0)

Agreement | confidence isn't  loud | English | 251.0s -> 301.0s | conf 0.76
out all the way you realize that's actually what life is it's not about you know I had a similar experience where I thought that achieving this goal getting this  ->  job or you know reaching this point was going to feel a certain way and yeah okay it felt great but then it was over and it was like okay


/usr/local/lib/python3.13/dist-packages/IPython/core/display.py:724: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


Open on YouTube (starts here, does not stop): https://www.youtube.com/watch?v=rKpltaOMFdc&t=251s


In [15]:
# Fallback: cut ONLY the segment (not the whole video) and play the mp4 inside Colab. Needs yt-dlp + ffmpeg (preinstalled in Colab).
def clip_and_play(i):
    import yt_dlp
    from yt_dlp.utils import download_range_func
    from IPython.display import Video
    r = results.iloc[i]; os.makedirs('clips', exist_ok=True); name = f'clips/{r.video_id}_{i}'
    opts = {'format': 'best[height<=480][ext=mp4]/best[height<=480]', 'quiet': True,
            'download_ranges': download_range_func(None, [(r.seg_start, r.seg_end)]),
            'force_keyframes_at_cuts': True, 'outtmpl': name + '.%(ext)s'}
    with yt_dlp.YoutubeDL(opts) as ydl: ydl.download([f'https://www.youtube.com/watch?v={r.yt_id}'])
    f = [x for x in os.listdir('clips') if x.startswith(f'{r.video_id}_{i}.')][0]
    return Video('clips/' + f, embed=True, width=640)
# clip_and_play(0)

## 10. Download the model (so the app can use it)
The model folder is ~1 GB. Save it to Google Drive or push it to the Hugging Face Hub; do **not** commit it to GitHub directly.

In [16]:
!zip -qr indicbert_moments.zip models/indicbert_moments
# from google.colab import drive; drive.mount('/content/drive'); !cp indicbert_moments.zip /content/drive/MyDrive/
# Or: from huggingface_hub import notebook_login; notebook_login(); clf.model.push_to_hub('YOUR_NAME/indicbert-moments'); clf.tok.push_to_hub('YOUR_NAME/indicbert-moments')

## 11. Streamlit web app
The app imports `pipeline.py`, loads the fine-tuned model, lets you browse results (with start-to-end playback) and analyse any new YouTube link.

In [17]:
%%writefile app.py
"""Streamlit app - multilingual key-moment finder (Question-Answer / Agreement / Disagreement).
Run:  streamlit run app.py
"""
import os
import pandas as pd
import streamlit as st
import streamlit.components.v1 as components
import pipeline as pl

st.set_page_config(page_title='Key-Moment Finder', page_icon='🎙️', layout='wide')

TYPE_ICON = {'Question-Answer': '❓', 'Agreement': '🤝', 'Disagreement': '⚡'}
LANGS = ['English', 'Hindi', 'Malayalam', 'Tamil', 'Urdu', 'Telugu', 'Kannada', 'Bengali', 'Marathi', 'Gujarati', 'Punjabi']


@st.cache_resource(show_spinner='Loading fine-tuned IndicBERT model...')
def load_clf(path):
    try:
        return pl.MomentClassifier(path)
    except Exception as e:                               # model folder missing / not downloaded
        return str(e)


def show_player(row, key):
    """Plays the segment from seg_start to seg_end."""
    s, e = int(row['seg_start']), int(row['seg_end'])
    url = f"https://www.youtube.com/watch?v={row['yt_id']}"
    try:
        st.video(url, start_time=s, end_time=e)
    except TypeError:                                    # very old Streamlit without end_time
        components.html(pl.yt_embed_html(row['yt_id'], s, e, w=640, h=360), height=380)
    st.link_button('Open on YouTube at this moment', row['watch_url'], key=f'lb_{key}')


def show_moment(row, key):
    icon = TYPE_ICON.get(row['type'], '')
    st.markdown(f"**{icon} {row['type']}** · moment at `{row['moment_time']}` · "
                f"segment {pl.fmt(row['seg_start'])}-{pl.fmt(row['seg_end'])} ({row['duration_s']:.0f}s) · "
                f"confidence {row['confidence']:.2f}")
    c1, c2 = st.columns([3, 2])
    with c1:
        show_player(row, key)
    with c2:
        st.caption('Key exchange (question/reply)')
        st.write(row['key_exchange'])
        with st.expander('Full segment transcript'):
            st.write(row['segment_transcript'])


# ------------------------------------------------------------------ sidebar
st.sidebar.title('🎙️ Key-Moment Finder')
model_dir = st.sidebar.text_input('Fine-tuned model folder / HF repo id',
                                  os.environ.get('MOMENT_MODEL', 'models/indicbert_moments'))
clf = load_clf(model_dir)
if isinstance(clf, str):
    st.sidebar.warning('Fine-tuned model not found -> using rule-based fallback.\n\n' + clf[:150])
    clf = None
else:
    st.sidebar.success('Fine-tuned IndicBERT loaded')

cfg = dict(pl.CFG)
cfg['buffer'] = st.sidebar.slider('Buffer y (seconds before/after)', 15, 30, 25,
                                  help='Segment = [x - y, x + y]; kept within 30-60 s')
cfg['ad_thr'] = st.sidebar.slider('Agree/Disagree confidence threshold', 0.5, 0.95, 0.70, 0.05)
cfg['q_thr'] = st.sidebar.slider('Question confidence threshold', 0.5, 0.95, 0.60, 0.05)

tab1, tab2, tab3 = st.tabs(['📂 Browse extracted moments', '🔎 Analyse a new video', 'ℹ️ About'])

# ------------------------------------------------------------------ tab 1
with tab1:
    up = st.file_uploader('Upload extracted_moments.csv (or leave empty to use the one in this folder)', type='csv')
    df = None
    if up is not None:
        df = pd.read_csv(up)
    elif os.path.exists('extracted_moments.csv'):
        df = pd.read_csv('extracted_moments.csv')
    if df is None or df.empty:
        st.info('No results yet. Run the notebook first, or use the "Analyse a new video" tab.')
    else:
        if 'language' not in df.columns:
            df['language'] = 'English'
        f1, f2, f3 = st.columns(3)
        langs = f1.multiselect('Language', sorted(df.language.unique()), default=sorted(df.language.unique()))
        types = f2.multiselect('Moment type', sorted(df.type.unique()), default=sorted(df.type.unique()))
        vids = f3.multiselect('Video', sorted(df.title.unique()), default=sorted(df.title.unique()))
        view = df[df.language.isin(langs) & df.type.isin(types) & df.title.isin(vids)].reset_index(drop=True)
        st.write(f'**{len(view)}** moments')
        st.dataframe(view[['video_id', 'language', 'type', 'moment_time', 'duration_s', 'confidence', 'key_exchange']],
                     use_container_width=True, height=250)
        st.download_button('Download filtered CSV', view.to_csv(index=False).encode('utf-8-sig'),
                           'moments_filtered.csv', 'text/csv')
        if len(view):
            labels = [f"{i}: {r.video_id} | {r.type} | {r.moment_time} | {str(r.title)[:35]}" for i, r in view.iterrows()]
            pick = st.selectbox('Pick a moment to play', range(len(view)), format_func=lambda i: labels[i])
            show_moment(view.iloc[pick], f'b{pick}')

# ------------------------------------------------------------------ tab 2
with tab2:
    st.write('Paste a YouTube link (podcast / interview / debate **with captions**), pick its spoken language, and run.')
    c1, c2 = st.columns([3, 1])
    url = c1.text_input('YouTube URL')
    lang = c2.selectbox('Spoken language', LANGS)
    if st.button('Find key moments', type='primary'):
        yt_id = pl.extract_video_id(url)
        if not yt_id:
            st.error('Could not read a video id from that URL.')
        else:
            with st.spinner('Fetching captions and classifying sentences...'):
                res, info = pl.analyse(yt_id, lang, title=url, video_id='LIVE', clf=clf, cfg=cfg)
            if res.empty:
                st.error('No captions available, or no moments found. Try another video, or add Whisper transcription.')
            else:
                if info.get('lang_used') and not str(info['lang_used']).lower().startswith(pl.LANG_CODES[pl.lang_key(lang)][0]):
                    st.warning(f"Captions were in '{info['lang_used']}', not {lang}. Results may be unreliable.")
                st.caption(f"{info['fragments']} caption fragments -> {info['sentences']} sentences · "
                           f"punctuated: {info['punctuated']} · sentence labels: {info['label_counts']}")
                st.session_state['live'] = res
    if 'live' in st.session_state:
        res = st.session_state['live']
        st.download_button('Download CSV', res.to_csv(index=False).encode('utf-8-sig'), 'live_moments.csv', 'text/csv')
        for i, r in res.iterrows():
            with st.expander(f"{TYPE_ICON.get(r['type'], '')} {r['type']} · {r['moment_time']} · conf {r['confidence']:.2f}",
                             expanded=(i == 0)):
                show_moment(r, f'l{i}')

# ------------------------------------------------------------------ tab 3
with tab3:
    st.markdown("""
**Pipeline** captions (no video download) -> sentences -> **IndicBERT (fine-tuned)** classifies each
(previous sentence, current sentence) pair as *neutral / question / agreement / disagreement* ->
moments -> segments `[x - y, x + y]` (30-60 s) -> playback.

**Languages** English, Hindi, Malayalam, Tamil, Urdu (IndicBERT v2 covers 20+ Indian languages; add more by
adding videos + labels).

**Limits** captions have no speaker labels, auto-captions can be wrong, and accuracy depends on the quality
and size of the labelled training data.
""")

Writing app.py


In [ ]:
# Run the app from Colab (public URL through localtunnel). On your own PC just run:  streamlit run app.py
!streamlit run app.py --server.port 8501 --server.headless true &>/content/streamlit.log &
!sleep 5; npm install -g localtunnel >/dev/null 2>&1
import urllib.request; print('Password (tunnel IP):', urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode().strip())
!npx localtunnel --port 8501

Password (tunnel IP): 34.127.37.64
⠙⠹⠸⠼your url is: https://proud-trees-rest.loca.lt

8501


## 12. Notes
- **No captions for a video?** Use Whisper (`faster-whisper`) to transcribe the audio: Hindi/Tamil/Urdu work reasonably, Malayalam is weaker. Output must be converted to `{'text','start','duration'}` snippets - then everything above works unchanged.
- **Romanised captions** (Hindi typed in English letters, "Hinglish") are not Devanagari, so IndicBERT and the word lists will not see them as Hindi. Choose videos whose captions are in the native script.
- **Weak spots to mention in the report:** captions have no speaker labels; timestamps within a caption are estimated; accuracy per language depends on how many reviewed labels you have; cue lists were written by me and need native-speaker review.